# E-commerce Shopping Intention Prediction using Decision Trees

### What was learned & implemented in this notebook:
1. **Exploratory Data Analysis (EDA) on Web Session Logs**:
   - Analyzed customer session metrics including Page Duration, Exit Rates, Bounce Rates, and Page Values.
2. **Data Preprocessing & Encoding**:
   - Implemented One-Hot Encoding (`OneHotEncoder`) for categorical variables like `VisitorType` and `Month` to make them suitable for scikit-learn models.
   - Managed boolean column conversion (`Weekend`, `Revenue`).
3. **Decision Tree Classification**:
   - Built a `DecisionTreeClassifier` with `max_depth` restriction to prevent overfitting.
   - Visualized the decision rules of the trained tree model using `plot_tree`.
4. **Model Performance Evaluation**:
   - Assessed accuracy (~89%), precision (~73%), recall (~52%), and F1-score (~61%) on the test split, highlighting how `PageValues` serves as a highly dominant predictor of purchase intention.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [12]:
df = pd.read_csv("shop_smart_ecommerce.csv")
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [24]:
df.sample(50)

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
7936,2,31.200000,0,0.00,17,2077.166667,0.000000,0.011765,0.000000,0.0,Nov,2,2,8,2,Returning_Visitor,True,False
9902,2,14.250000,0,0.00,29,1082.750000,0.000000,0.006897,84.875727,0.0,Nov,2,5,3,2,New_Visitor,False,True
6890,3,10.000000,0,0.00,2,8.000000,0.000000,0.033333,0.000000,0.0,Sep,2,2,1,2,New_Visitor,False,True
11198,1,96.666667,0,0.00,23,1605.875000,0.031818,0.050303,54.102308,0.0,Dec,2,2,2,10,Returning_Visitor,False,True
12017,6,1922.000000,0,0.00,30,941.672619,0.018182,0.044444,0.000000,0.0,Nov,3,2,2,1,Returning_Visitor,False,False
1968,0,0.000000,0,0.00,32,999.000000,0.000000,0.013441,9.053082,0.0,Mar,1,1,1,1,Returning_Visitor,False,True
11059,1,12.000000,2,11.00,18,974.166667,0.005455,0.032251,0.000000,0.0,Nov,3,2,1,2,New_Visitor,False,False
1121,0,0.000000,0,0.00,12,200.000000,0.000000,0.047222,0.000000,0.0,Mar,2,2,1,2,Returning_Visitor,True,False
4204,1,0.000000,0,0.00,7,110.750000,0.000000,0.028571,0.000000,0.0,May,3,2,1,4,Returning_Visitor,True,False
3466,0,0.000000,0,0.00,37,1669.559524,0.016190,0.051333,0.000000,0.0,May,3,2,3,1,Returning_Visitor,False,False


In [9]:
df.isnull().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [11]:
df.shape

(12330, 18)

In [23]:
ohe= OneHotEncoder(drop="first",sparse_output = False,handle_unknown="ignore")
encoded = ohe.fit_transform(df[["VisitorType"]])
encoded_df = pd.DataFrame(encoded,index= df.index,columns=ohe.get_feature_names_out(["VisitorType"]))
#encoded_df = encoded_df.drop(["VisitorType_Other"])
encoded_df


,VisitorType_Other,VisitorType_Returning_Visitor
0,0.0,1.0
1,0.0,1.0
2,0.0,1.0
3,0.0,1.0
4,0.0,1.0
...,...,...
12325,0.0,1.0
12326,0.0,1.0
12327,0.0,1.0
12328,0.0,1.0


### Full Data Preprocessing, Model Training & Evaluation
We will now preprocess all categorical columns (VisitorType and Month), split the dataset, fit a `DecisionTreeClassifier`, and evaluate its performance.

In [25]:
# 1. One-hot encode VisitorType and Month
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoded_cats = ohe.fit_transform(df[['VisitorType', 'Month']])
encoded_cats_df = pd.DataFrame(encoded_cats, index=df.index, columns=ohe.get_feature_names_out(['VisitorType', 'Month']))

# Combine with numerical features
df_preprocessed = df.drop(columns=['VisitorType', 'Month'])
df_preprocessed = pd.concat([df_preprocessed, encoded_cats_df], axis=1)

# Convert Weekend and Revenue to integers
df_preprocessed['Weekend'] = df_preprocessed['Weekend'].astype(int)
df_preprocessed['Revenue'] = df_preprocessed['Revenue'].astype(int)

# Split features X and target y
X = df_preprocessed.drop(columns=['Revenue'])
y = df_preprocessed['Revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [26]:
# 2. Initialize and fit the DecisionTreeClassifier
dt_classifier = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_classifier.fit(X_train, y_train)

# Predict
y_pred = dt_classifier.predict(X_test)

# Evaluation
print('Decision Tree Classifier (max_depth=4)')
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall:', round(recall_score(y_test, y_pred), 4))
print('F1 Score:', round(f1_score(y_test, y_pred), 4))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))

Decision Tree Classifier (max_depth=4)
Accuracy: 0.89
Precision: 0.73
Recall: 0.52
F1 Score: 0.61


In [27]:
# 3. Plot the Decision Tree
import matplotlib.pyplot as plt
plt.figure(figsize=(20, 10))
plot_tree(dt_classifier, max_depth=2, feature_names=list(X.columns), class_names=['No Revenue', 'Revenue'], filled=True, fontsize=10)
plt.show()